# Kafka 与流处理 (Kafka & Streaming)

## 高级数据工程师面试核心考点

本节涵盖高频考点：
- Topic / Partition / Consumer Group
- Offset 管理 & At-least-once / Exactly-once
- Replication Factor & ISR (In-Sync Replicas)
- Log Compaction (日志压缩)
- Consumer Lag 监控
- Kafka Connect & Schema Registry

---
> **面试提示**：Kafka 是大数据工程中最核心的消息系统，几乎所有大厂都在使用。理解其底层原理（而非只会调 API）是高级工程师的必备技能。

---
## 1. Topic / Partition / Consumer Group (高频)

### 核心架构

```
Kafka 整体架构图：

  生产者 (Producers)            Kafka Cluster              消费者 (Consumers)
  ┌──────────┐              ┌─────────────────────┐
  │Producer 1│──────────→  │ Topic: user-events  │
  └──────────┘             │ ┌─────────────────┐ │  Consumer Group A
  ┌──────────┐             │ │ Partition 0     │ │──→ ┌────────────┐
  │Producer 2│──────────→  │ │ [0][1][2][3][4] │ │   │ Consumer 1 │ (读 P0)
  └──────────┘             │ ├─────────────────┤ │   └────────────┘
  ┌──────────┐             │ │ Partition 1     │ │──→ ┌────────────┐
  │Producer 3│──────────→  │ │ [0][1][2][3]   │ │   │ Consumer 2 │ (读 P1)
  └──────────┘             │ ├─────────────────┤ │   └────────────┘
                           │ │ Partition 2     │ │──→ ┌────────────┐
                           │ │ [0][1][2][3][4] │ │   │ Consumer 3 │ (读 P2)
                           │ └─────────────────┘ │   └────────────┘
                           └─────────────────────┘
                                                    Consumer Group B
                                                    ┌────────────┐
                                                    │ Consumer 4 │ (读 P0+P1)
                                                    ├────────────┤
                                                    │ Consumer 5 │ (读 P2)
                                                    └────────────┘

核心规则（面试必背）：
  1. 同一 Consumer Group 内，一个 Partition 只能被一个 Consumer 消费
  2. 不同 Consumer Group 可以同时消费同一 Topic（独立 Offset）
  3. Consumer 数量 > Partition 数量 → 多余的 Consumer 空闲（无法消费）
  4. Partition = 并行度单位 → 提高吞吐量需增加 Partition 数
  5. Partition 内消息有序，Topic 级别无序

Partition Key 选择：
  - 同一 key 的消息总是发往同一 Partition → 保证 key 级别有序
  - user_id 作为 key → 同一用户的所有事件有序
  - 选择高基数 key 避免热点（不要用 country_code 只有 200 个值）
  - null key → 轮询（Round Robin）分配
```

In [ ]:
# Simulate Kafka Topic/Partition/Consumer Group concepts in Python
import hashlib
from collections import defaultdict

class KafkaTopic:
    """Simplified Kafka Topic simulation."""
    
    def __init__(self, name: str, num_partitions: int):
        self.name = name
        self.num_partitions = num_partitions
        self.partitions = [[] for _ in range(num_partitions)]  # Each partition is a list of messages
        self.offsets = [0] * num_partitions  # Next offset for each partition
    
    def _get_partition(self, key: str) -> int:
        """Consistent hash-based partition assignment (same as Kafka's default partitioner)."""
        if key is None:
            import random
            return random.randint(0, self.num_partitions - 1)
        hash_val = int(hashlib.md5(key.encode()).hexdigest(), 16)
        return hash_val % self.num_partitions
    
    def produce(self, key: str, value: str) -> tuple:
        """Produce a message to the appropriate partition."""
        partition = self._get_partition(key)
        offset = self.offsets[partition]
        self.partitions[partition].append({'key': key, 'value': value, 'offset': offset})
        self.offsets[partition] += 1
        return partition, offset
    
    def print_state(self):
        print(f"\nTopic: {self.name} ({self.num_partitions} partitions)")
        for i, partition in enumerate(self.partitions):
            msgs = [f"[{m['offset']}]{m['key']}={m['value']}" for m in partition]
            print(f"  P{i}: {' '.join(msgs) if msgs else '(empty)'}")


class ConsumerGroup:
    """Simulate a Kafka Consumer Group with partition assignment."""
    
    def __init__(self, group_id: str, num_consumers: int, topic: KafkaTopic):
        self.group_id = group_id
        self.num_consumers = num_consumers
        self.topic = topic
        self.committed_offsets = defaultdict(int)  # partition_id -> committed offset
        self.assignment = self._assign_partitions()
    
    def _assign_partitions(self) -> dict:
        """Range assignment strategy: distribute partitions across consumers."""
        assignment = defaultdict(list)  # consumer_id -> [partition_ids]
        n_p = self.topic.num_partitions
        n_c = self.num_consumers
        for p in range(n_p):
            consumer = p % n_c  # Simple round-robin assignment
            assignment[consumer].append(p)
        return assignment
    
    def print_assignment(self):
        print(f"\nConsumer Group: {self.group_id} ({self.num_consumers} consumers)")
        for consumer_id in range(self.num_consumers):
            partitions = self.assignment.get(consumer_id, [])
            if partitions:
                print(f"  Consumer-{consumer_id}: owns Partitions {partitions}")
            else:
                print(f"  Consumer-{consumer_id}: IDLE (no partition assigned!)")


# Demo
topic = KafkaTopic('user-events', num_partitions=3)

# Produce messages with different keys
messages = [
    ('user_001', 'login'), ('user_002', 'click'), ('user_003', 'purchase'),
    ('user_001', 'logout'), ('user_004', 'login'), ('user_002', 'view'),
    ('user_005', 'click'), ('user_001', 'click'),  # user_001 always goes to same partition
]
print("=== Producing Messages ===")
for key, value in messages:
    partition, offset = topic.produce(key, value)
    print(f"  key={key}, value={value} → Partition {partition}, Offset {offset}")

topic.print_state()

print("\n=== Consumer Group Assignments ===")
# Group A: 3 consumers, 3 partitions → perfect balance
group_a = ConsumerGroup('group-A', num_consumers=3, topic=topic)
group_a.print_assignment()

# Group B: 2 consumers, 3 partitions → one consumer gets 2 partitions
group_b = ConsumerGroup('group-B', num_consumers=2, topic=topic)
group_b.print_assignment()

# Group C: 4 consumers, 3 partitions → one consumer is idle!
group_c = ConsumerGroup('group-C', num_consumers=4, topic=topic)
group_c.print_assignment()

print("\n[Key Insight] Consumer > Partition → idle consumer! Adding partitions increases parallelism.")

---
## 2. Offset 管理 & 消息传递语义 (高频)

### 三种传递语义

```
消息传递语义对比：

┌──────────────────┬───────────────────────────────────────────────────────────┐
│ 语义             │ 实现方式                                                   │
├──────────────────┼───────────────────────────────────────────────────────────┤
│ At-most-once     │ 消费前先提交 offset（消息可能丢失，但绝不重复）              │
│ (最多一次)       │ consumer.poll() → commit() → process()                    │
│                  │ 用途：允许丢失的日志/指标场景                               │
├──────────────────┼───────────────────────────────────────────────────────────┤
│ At-least-once    │ 处理后提交 offset（消息可能重复，但绝不丢失）               │
│ (至少一次)       │ consumer.poll() → process() → commit()                    │
│                  │ 崩溃重启后从上次提交 offset 重新消费                       │
│                  │ 需要下游幂等（idempotent）处理                             │
│                  │ 用途：大多数数据管道                                        │
├──────────────────┼───────────────────────────────────────────────────────────┤
│ Exactly-once     │ 幂等生产者 + 事务 API                                      │
│ (精确一次)       │ Producer: enable.idempotence=true（PID + 序列号去重）      │
│                  │ Transactions: beginTransaction → send → commitTransaction │
│                  │ Consumer: isolation.level=read_committed                  │
│                  │ 用途：金融交易、精确计费                                    │
└──────────────────┴───────────────────────────────────────────────────────────┘

Offset 提交方式：
  自动提交：enable.auto.commit=true，auto.commit.interval.ms=5000
             风险：处理中崩溃 → 已提交但未处理的消息丢失（at-most-once 风险）

  手动提交（同步）：consumer.commitSync()
             更安全，确保消息处理完成后才提交

  手动提交（异步）：consumer.commitAsync(callback)
             非阻塞，但失败可能导致乱序提交（要小心）

  精确 offset 提交：commitSync({partition: offset})
             最精细控制，适合批处理完成后的提交
```

In [ ]:
# Simulate different offset commit strategies and their failure behavior
import time

class MockKafkaConsumer:
    """Mock consumer to demonstrate offset commit strategies."""
    
    def __init__(self, messages, simulate_crash_at=None):
        self.messages = messages
        self.current_index = 0
        self.committed_offset = 0  # Last committed offset
        self.simulate_crash_at = simulate_crash_at  # Offset to crash at
        self.crashes = 0
    
    def poll(self, batch_size=3):
        """Poll for the next batch of messages."""
        batch = self.messages[self.current_index:self.current_index + batch_size]
        self.current_index += len(batch)
        return batch
    
    def commit(self, offset=None):
        """Commit the given offset (or current position)."""
        self.committed_offset = offset or self.current_index
    
    def simulate_failure_at_offset(self, offset):
        """Check if we should simulate a crash."""
        if self.simulate_crash_at and offset >= self.simulate_crash_at and self.crashes == 0:
            self.crashes += 1
            self.current_index = self.committed_offset  # Reset to last committed
            return True
        return False

messages = [f'msg_{i}' for i in range(10)]

# Strategy 1: At-most-once (commit BEFORE processing)
print("=== Strategy 1: At-most-once (commit before process) ===")
consumer = MockKafkaConsumer(messages.copy(), simulate_crash_at=4)
processed = []
for batch_num in range(4):
    batch = consumer.poll(batch_size=3)
    if not batch:
        break
    consumer.commit()  # Commit BEFORE processing
    print(f"  Committed offset: {consumer.committed_offset}")
    crashed = consumer.simulate_failure_at_offset(consumer.current_index)
    if crashed:
        print(f"  CRASH! Reset to committed offset: {consumer.committed_offset}")
        print(f"  Messages {messages[3:6]} were LOST! (committed but never processed)")
        break
    for msg in batch:
        processed.append(msg)
        print(f"    Processed: {msg}")

print()

# Strategy 2: At-least-once (commit AFTER processing)
print("=== Strategy 2: At-least-once (commit after process) ===")
consumer2 = MockKafkaConsumer(messages.copy(), simulate_crash_at=5)
processed2 = []
crashed = False
for batch_num in range(5):
    batch = consumer2.poll(batch_size=2)
    if not batch:
        break
    current_offset = consumer2.current_index
    crashed = consumer2.simulate_failure_at_offset(current_offset)
    if crashed:
        print(f"  CRASH at offset {current_offset}! Reset to last committed: {consumer2.committed_offset}")
        # Re-consume from committed offset
        re_batch = messages[consumer2.committed_offset:consumer2.committed_offset + 2]
        print(f"  Re-consuming (DUPLICATE): {re_batch}")
        print(f"  → Messages are DUPLICATED but never lost!")
        break
    for msg in batch:
        processed2.append(msg)
        print(f"    Processed: {msg}")
    consumer2.commit()  # Commit AFTER processing
    print(f"  Committed offset: {consumer2.committed_offset}")

print()
print("=== Strategy 3: Exactly-once (conceptual) ===")
print("  Producer: enable.idempotence=true")
print("    - Each message gets (ProducerID, SequenceNumber)")
print("    - Broker deduplicates based on (PID, SeqNum)")
print("  Transactions:")
print("    producer.initTransactions()")
print("    producer.beginTransaction()")
print("    producer.send(topic, record)")
print("    producer.sendOffsetsToTransaction(offsets, consumerGroupId)")
print("    producer.commitTransaction()  # Atomic: output + offset commit")
print("  Consumer: isolation.level=read_committed")
print("    - Only reads messages from committed transactions")

---
## 3. Replication Factor & ISR (重要)

### 高可用机制

```
Kafka 副本机制（Replication Factor = 3）：

  Broker 1 (Leader P0)     Broker 2 (Follower P0)    Broker 3 (Follower P0)
  ┌────────────────────┐   ┌────────────────────┐    ┌────────────────────┐
  │ Partition 0        │   │ Partition 0        │    │ Partition 0        │
  │ Leader             │──→│ Follower (replica) │──→ │ Follower (replica) │
  │ [0][1][2][3][4][5] │   │ [0][1][2][3][4]   │    │ [0][1][2][3]       │
  └────────────────────┘   └────────────────────┘    └────────────────────┘
        ↑
  Producer writes here
  Consumer reads here

ISR (In-Sync Replicas)：
  ISR = 与 Leader 保持同步的副本集合（包括 Leader 本身）
  Broker 2 在 ISR 中（延迟 1 条）
  Broker 3 落后太多 → 从 ISR 中移除

关键配置：
  replication.factor = 3          (每个 Partition 有 3 个副本)
  min.insync.replicas = 2         (至少 2 个副本确认写入才算成功)
  acks = all (或 -1)              (Producer 等待所有 ISR 副本确认)

故障场景分析：
  Leader Broker 1 宕机：
    1. Controller (Kafka 内部协调者) 检测到 Leader 下线
    2. 从 ISR 中选出新 Leader（Broker 2，因为它最新）
    3. 生产者和消费者自动重连到新 Leader
    4. Broker 1 恢复后作为 Follower 同步数据

  数据丢失风险：
    acks=1（只等 Leader 确认）：Leader 宕机后未同步到 Follower 的数据丢失
    acks=all + min.insync.replicas=2：最多 N-min.insync 个 Broker 同时宕机仍不丢数据

可用性 vs 持久性权衡：
  acks=0：最高吞吐，可能丢数据（fire-and-forget）
  acks=1：平衡，Leader 确认即可
  acks=all：最高持久性，延迟最高（等所有 ISR 确认）
```

In [ ]:
# Simulate ISR (In-Sync Replicas) and leader election
import time

class KafkaBroker:
    def __init__(self, broker_id: int, is_leader: bool = False):
        self.broker_id = broker_id
        self.is_leader = is_leader
        self.is_alive = True
        self.log_end_offset = 0  # Highest offset in this replica
        self.high_watermark = 0   # Highest offset confirmed by all ISR
    
    def replicate(self, leader_offset: int, lag_allowed: int = 0):
        """Simulate follower replicating from leader (with optional lag)."""
        self.log_end_offset = max(0, leader_offset - lag_allowed)


class KafkaPartition:
    """Simulate a Kafka partition with replication and ISR management."""
    
    def __init__(self, brokers: list, isr_lag_threshold: int = 2):
        self.brokers = {b.broker_id: b for b in brokers}
        self.leader_id = next(b.broker_id for b in brokers if b.is_leader)
        self.isr_lag_threshold = isr_lag_threshold
    
    @property
    def leader(self):
        return self.brokers[self.leader_id]
    
    def get_isr(self) -> list:
        """Return list of in-sync replicas (within lag threshold of leader)."""
        leader_offset = self.leader.log_end_offset
        return [
            b for b in self.brokers.values()
            if b.is_alive and (leader_offset - b.log_end_offset) <= self.isr_lag_threshold
        ]
    
    def produce(self, message: str, acks: str = 'all', min_insync: int = 2) -> bool:
        """Produce a message with specified ack level."""
        if not self.leader.is_alive:
            print(f"  ERROR: Leader {self.leader_id} is down!")
            return False
        
        self.leader.log_end_offset += 1
        new_offset = self.leader.log_end_offset
        
        if acks == '0':
            return True  # Fire and forget
        elif acks == '1':
            return True  # Leader acknowledged
        elif acks == 'all':
            isr = self.get_isr()
            if len(isr) < min_insync:
                print(f"  ERROR: ISR size {len(isr)} < min.insync.replicas {min_insync}")
                return False
            # Simulate ISR followers replicating
            for b in isr:
                if not b.is_leader:
                    b.log_end_offset = new_offset
            return True
    
    def elect_new_leader(self):
        """Elect a new leader from ISR when current leader fails."""
        isr = self.get_isr()
        # Find the most up-to-date follower in ISR
        candidates = [b for b in isr if not b.is_leader]
        if not candidates:
            print("  FATAL: No ISR candidates for leader election! Data may be lost.")
            return
        # Select broker with highest LEO (Log End Offset)
        new_leader = max(candidates, key=lambda b: b.log_end_offset)
        print(f"  Electing new leader: Broker {new_leader.broker_id} (LEO={new_leader.log_end_offset})")
        self.brokers[self.leader_id].is_leader = False
        new_leader.is_leader = True
        self.leader_id = new_leader.broker_id
    
    def print_state(self):
        isr = self.get_isr()
        isr_ids = [b.broker_id for b in isr]
        print(f"  Leader: Broker {self.leader_id} (LEO={self.leader.log_end_offset})")
        print(f"  ISR:    {isr_ids}")
        for bid, b in sorted(self.brokers.items()):
            status = 'LEADER' if b.is_leader else 'FOLLOWER'
            alive = 'UP' if b.is_alive else 'DOWN'
            in_isr = 'ISR' if b in isr else 'OUT-OF-ISR'
            print(f"    Broker {bid} [{status}][{alive}][{in_isr}]: LEO={b.log_end_offset}")


# Setup: 3 brokers, Broker 1 is leader
b1 = KafkaBroker(1, is_leader=True)
b2 = KafkaBroker(2)
b3 = KafkaBroker(3)
partition = KafkaPartition([b1, b2, b3])

print("=== Initial State ===")
partition.print_state()

# Produce 5 messages
print("\n=== Producing 5 messages (acks=all) ===")
for i in range(5):
    ok = partition.produce(f'msg_{i}', acks='all', min_insync=2)
    # Simulate Broker 3 lagging behind
    b3.replicate(b1.log_end_offset, lag_allowed=3)
    print(f"  Produced msg_{i}: {'OK' if ok else 'FAILED'} | B1.LEO={b1.log_end_offset}, B2.LEO={b2.log_end_offset}, B3.LEO={b3.log_end_offset}")

print("\n=== State after 5 messages (Broker 3 lagged) ===")
partition.print_state()

print("\n=== Simulating Leader (Broker 1) Failure ===")
b1.is_alive = False
partition.elect_new_leader()
print("\n=== State after leader election ===")
partition.print_state()

---
## 4. Log Compaction (重要)

### 日志压缩原理

```
Kafka 两种 Retention 策略：

  1. 基于时间/大小的删除（默认）：
     超过 retention.ms 或 retention.bytes 的 Segment 直接删除
     用途：日志、指标流水 → 只关心最近数据

  2. Log Compaction（日志压缩）：
     cleanup.policy=compact
     只保留每个 key 的最新值，删除历史旧值
     用途：变更事件流（CDC）、配置存储、数据库备份

Log Compaction 工作原理：

  压缩前（Partition 中的消息）：
  Offset:  0     1     2     3     4     5     6     7     8
  Key:     A     B     A     C     B     A     D     B     C
  Value:   v1    v1    v2    v1    v2    v3   v1    v3    v2

  压缩后（只保留每个 key 的最新值）：
  Offset:  5     6     7     8
  Key:     A     D     B     C
  Value:   v3    v1    v3    v2
  
  (A的offset 0,2 已删；B的offset 1,4已删；C的offset 3已删)

Tombstone 记录（删除标记）：
  value=null 的消息 = Tombstone
  表示该 key 被删除，压缩时会清除该 key 的所有历史记录
  Offset:  9
  Key:     D
  Value:   null  ← Tombstone，表示 D 被删除
  压缩后：D 的所有记录（包括 offset 6 和 9）都会被删除

主要配置：
  log.cleanup.policy = compact
  log.segment.bytes = 1GB      (Segment 大小)
  min.compaction.lag.ms        (消息至少存活多久才能被压缩)
  delete.retention.ms = 86400000  (Tombstone 保留时间，确保消费者看到删除)

典型使用场景：
  - Kafka as Database: 存储 key-value 状态的最新版本
  - CDC (Change Data Capture): MySQL binlog → Kafka compact topic → 下游数据库
  - Consumer 重启后只需从头读 compact topic 即可重建最新状态
```

In [ ]:
# Simulate Log Compaction
from collections import OrderedDict

def simulate_log_compaction(messages: list) -> list:
    """
    Simulate Kafka log compaction.
    Input: list of (offset, key, value) tuples
    Output: compacted list keeping only the latest value per key
    """
    # Find the latest offset for each key
    latest = {}  # key -> (offset, value)
    for offset, key, value in messages:
        latest[key] = (offset, value)
    
    # Build compacted log (sorted by offset, exclude tombstones with null value)
    compacted = [
        (offset, key, value)
        for key, (offset, value) in sorted(latest.items(), key=lambda x: x[1][0])
        if value is not None  # Remove tombstones
    ]
    
    # Also track tombstones that need to survive delete.retention.ms
    tombstones = [
        (offset, key, value)
        for key, (offset, value) in latest.items()
        if value is None
    ]
    
    return compacted, tombstones

# Simulate a compactable topic (e.g., user profile updates)
messages = [
    (0,  'user:1001', {'name': 'Alice', 'age': 28}),
    (1,  'user:1002', {'name': 'Bob', 'age': 35}),
    (2,  'user:1001', {'name': 'Alice', 'age': 29}),   # Alice updated age
    (3,  'user:1003', {'name': 'Carol', 'age': 42}),
    (4,  'user:1002', {'name': 'Bob', 'age': 35, 'email': 'bob@x.com'}),  # Bob added email
    (5,  'user:1001', {'name': 'Alice Smith', 'age': 29}),  # Alice updated name
    (6,  'user:1004', {'name': 'Dave', 'age': 31}),
    (7,  'user:1003', None),  # Carol deleted (Tombstone!)
    (8,  'user:1002', {'name': 'Bob', 'age': 36, 'email': 'bob@x.com'}),  # Bob updated age
    (9,  'user:1004', {'name': 'Dave', 'age': 32}),
]

print("=== Original Partition (10 messages) ===")
for offset, key, value in messages:
    print(f"  [{offset:2d}] {key}: {value}")

compacted, tombstones = simulate_log_compaction(messages)

print(f"\n=== After Log Compaction ({len(compacted)} messages retained) ===")
for offset, key, value in compacted:
    print(f"  [{offset:2d}] {key}: {value}")

print(f"\n=== Tombstones (pending delete.retention.ms) ===")
for offset, key, value in tombstones:
    print(f"  [{offset:2d}] {key}: null  ← will be removed after retention expires")

saved = len(messages) - len(compacted) - len(tombstones)
print(f"\nSpace saving: {saved}/{len(messages)} messages removed ({saved/len(messages)*100:.0f}%)")

---
## 5. Consumer Lag 监控 (高频)

### Consumer Lag 定义与计算

```
Consumer Lag 计算：
  Lag = Log End Offset (LEO) - Consumer Committed Offset

  示例：
  Partition 0: LEO=1000, Consumer Offset=950 → Lag=50
  Partition 1: LEO=2000, Consumer Offset=1800 → Lag=200
  Partition 2: LEO=500,  Consumer Offset=500  → Lag=0
  Total Lag = 50 + 200 + 0 = 250

命令行查看 lag：
  kafka-consumer-groups.sh --bootstrap-server kafka:9092 \
    --describe --group my-consumer-group

  GROUP           TOPIC          PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG
  my-group        user-events    0          950             1000            50
  my-group        user-events    1          1800            2000            200
  my-group        user-events    2          500             500             0

监控策略：
  指标           告警阈值（参考）      说明
  Total Lag      > 10,000            积压严重
  Lag 增长速率    > 0 (持续增长)      消费速度跟不上生产速度
  Max Partition  某分区 >> 其他      分区热点或 Consumer 失活
  Consumer Count  < 期望值           Consumer 崩溃未恢复

Lag 增大的常见原因：
  1. 消费者处理速度 < 生产者写入速度
  2. 下游系统（DB、API）变慢，消费阻塞
  3. GC 暂停（Java 消费者）
  4. 某个 Consumer 实例挂掉，触发 Rebalance 导致短暂停止消费
  5. 网络分区导致 Consumer 无法连接 Broker

解决方案：
  - 增加 Consumer 实例（不能超过 Partition 数）
  - 增加 Partition 数（需要重建 Consumer Group）
  - 优化消费者处理逻辑（批量处理、异步写入）
  - 增加 fetch.max.bytes 减少 poll 次数
```

In [ ]:
# Simulate Consumer Lag monitoring and alerting
import random
import time
from collections import defaultdict

class LagMonitor:
    """Simulate Kafka consumer lag monitoring."""
    
    def __init__(self, topic: str, partitions: int, lag_alert_threshold: int = 1000):
        self.topic = topic
        self.partitions = partitions
        self.lag_alert_threshold = lag_alert_threshold
        # Simulate LEO and consumer offsets
        self.leo = {p: 5000 for p in range(partitions)}  # Log End Offset
        self.consumer_offset = {p: 5000 for p in range(partitions)}
        self.history = []  # (timestamp, total_lag)
    
    def simulate_production(self, msgs_per_second: dict):
        """Simulate producers writing messages."""
        for p, rate in msgs_per_second.items():
            self.leo[p] += rate
    
    def simulate_consumption(self, msgs_per_second: dict):
        """Simulate consumers processing messages."""
        for p, rate in msgs_per_second.items():
            self.consumer_offset[p] = min(
                self.consumer_offset[p] + rate,
                self.leo[p]  # Can't consume beyond LEO
            )
    
    def get_lag_report(self) -> dict:
        lag_per_partition = {p: self.leo[p] - self.consumer_offset[p] for p in range(self.partitions)}
        total_lag = sum(lag_per_partition.values())
        max_partition = max(lag_per_partition, key=lag_per_partition.get)
        return {
            'total': total_lag,
            'per_partition': lag_per_partition,
            'max_partition': max_partition,
            'max_lag': lag_per_partition[max_partition],
        }
    
    def print_report(self, label: str):
        report = self.get_lag_report()
        alert = " *** ALERT ***" if report['total'] > self.lag_alert_threshold else ""
        print(f"\n[{label}]{alert}")
        print(f"  {'PARTITION':<12} {'LEO':>10} {'COMMITTED':>12} {'LAG':>8}")
        print(f"  {'-'*45}")
        for p in range(self.partitions):
            lag = self.leo[p] - self.consumer_offset[p]
            flag = " ← HOT" if lag == report['max_lag'] and lag > 0 else ""
            print(f"  P{p:<11} {self.leo[p]:>10} {self.consumer_offset[p]:>12} {lag:>8}{flag}")
        print(f"  {'TOTAL':>35} {report['total']:>8}")


# Scenario: 3-partition topic, consumer can't keep up with producer
monitor = LagMonitor('user-events', partitions=3, lag_alert_threshold=500)

# Tick 1: Normal operation
monitor.simulate_production({0: 100, 1: 100, 2: 100})
monitor.simulate_consumption({0: 100, 1: 100, 2: 100})
monitor.print_report("Tick 1: Normal operation")

# Tick 2: Production spikes (e.g., marketing campaign)
monitor.simulate_production({0: 500, 1: 500, 2: 500})
monitor.simulate_consumption({0: 100, 1: 100, 2: 100})
monitor.print_report("Tick 2: Production spike")

# Tick 3: Partition 1 consumer dies (rebalancing - consumption paused)
monitor.simulate_production({0: 200, 1: 200, 2: 200})
monitor.simulate_consumption({0: 200, 1: 0, 2: 200})  # P1 consumer is down!
monitor.print_report("Tick 3: Partition 1 consumer down")

# Tick 4: Recovery
monitor.simulate_production({0: 200, 1: 200, 2: 200})
monitor.simulate_consumption({0: 200, 1: 800, 2: 200})  # P1 catching up!
monitor.print_report("Tick 4: Recovery (P1 catching up)")

---
## 6. Kafka Connect & Schema Registry (重要)

### Kafka Connect 架构

```
Kafka Connect 数据流：

  数据源                Kafka Connect              Kafka             目标系统
  ┌─────────┐          ┌─────────────┐           ┌────────┐        ┌──────────┐
  │ MySQL   │─Source──→│ JDBC Source │──produce→ │ Topic  │        │          │
  │ Postgres│  Connector│ Connector  │           │        │        │          │
  │ MongoDB │          └─────────────┘           │ user-  │─Sink──→│ BigQuery │
  └─────────┘                                    │ events │ Connector│ S3      │
                                                 │        │        │ Snowflake│
  ┌─────────┐          ┌─────────────┐           └────────┘        └──────────┘
  │ File    │─Source──→│ File Source │──produce→
  │ S3      │  Connector│ Connector  │
  └─────────┘          └─────────────┘

运行模式：
  Standalone: 单节点，适合开发测试
  Distributed: 多节点，生产使用，自动负载均衡和故障恢复

Connector 配置（示例）：
  {
    "name": "mysql-source",
    "config": {
      "connector.class": "io.confluent.connect.jdbc.JdbcSourceConnector",
      "connection.url": "jdbc:mysql://db:3306/mydb",
      "table.whitelist": "users,orders",
      "mode": "timestamp+incrementing",
      "timestamp.column.name": "updated_at",
      "incrementing.column.name": "id",
      "topic.prefix": "db-",
      "value.converter": "io.confluent.kafka.serializers.KafkaAvroSerializer",
      "value.converter.schema.registry.url": "http://schema-registry:8081"
    }
  }

Schema Registry 兼容性模式：
  BACKWARD:   新 Schema 可以读旧数据（加字段有默认值，删字段）
              消费者先升级，生产者后升级
  FORWARD:    旧 Schema 可以读新数据（加字段，删带默认值的字段）
              生产者先升级，消费者后升级
  FULL:       同时满足 BACKWARD 和 FORWARD
              最安全，要求最严格
  NONE:       不检查兼容性

Schema Registry 工作流：
  生产者: serialize(data, schema) → 注册 Schema → 发送 (schema_id + binary_data)
  消费者: 收到消息 → 读取 schema_id → 从 Registry 获取 Schema → deserialize
  好处:   消息体不含 Schema（仅 4 字节 schema_id），极大减小消息体积
```

In [ ]:
# Simulate Kafka Connect configuration and Schema Registry behavior
import json
import struct
import hashlib

# ── Kafka Connect Config Examples ───────────────────────────────────────────
source_connector_config = {
    "name": "mysql-cdc-source",
    "config": {
        "connector.class": "io.debezium.connector.mysql.MySqlConnector",
        "database.hostname": "mysql",
        "database.port": "3306",
        "database.user": "debezium",
        "database.password": "${file:/secrets.properties:db.password}",
        "database.server.id": "184054",
        "database.server.name": "mydb",
        "table.include.list": "mydb.users,mydb.orders",
        "database.history.kafka.bootstrap.servers": "kafka:9092",
        "database.history.kafka.topic": "dbhistory.mydb",
        "transforms": "route",
        "transforms.route.type": "org.apache.kafka.connect.transforms.ReplaceField$Value",
        "value.converter": "io.confluent.kafka.serializers.KafkaAvroSerializer",
        "value.converter.schema.registry.url": "http://schema-registry:8081"
    }
}

sink_connector_config = {
    "name": "s3-sink",
    "config": {
        "connector.class": "io.confluent.connect.s3.S3SinkConnector",
        "tasks.max": "4",
        "topics": "mydb.users,mydb.orders",
        "s3.region": "us-east-1",
        "s3.bucket.name": "my-data-lake",
        "s3.part.size": "67108864",
        "flush.size": "10000",
        "rotate.interval.ms": "3600000",
        "storage.class": "io.confluent.connect.s3.storage.S3Storage",
        "format.class": "io.confluent.connect.s3.format.parquet.ParquetFormat",
        "parquet.codec": "snappy",
        "locale": "US",
        "timezone": "UTC",
        "timestamp.extractor": "RecordField",
        "timestamp.field": "created_at",
        "partitioner.class": "io.confluent.connect.storage.partitioner.TimeBasedPartitioner",
        "path.format": "'year'=YYYY/'month'=MM/'day'=dd",
    }
}

print("=== Source Connector Config (Debezium MySQL CDC) ===")
print(json.dumps(source_connector_config, indent=2))

print("\n=== Sink Connector Config (S3 Parquet Sink) ===")
print(json.dumps(sink_connector_config, indent=2))

In [ ]:
# Simulate Schema Registry: schema registration and compatibility checking

class SchemaRegistry:
    """Simplified Schema Registry simulation."""
    
    def __init__(self, compatibility: str = 'BACKWARD'):
        self.schemas = {}  # subject -> list of schemas (versions)
        self.schema_id_map = {}  # schema_hash -> schema_id
        self.next_id = 1
        self.compatibility = compatibility
    
    def _schema_hash(self, schema: dict) -> str:
        return hashlib.md5(json.dumps(schema, sort_keys=True).encode()).hexdigest()[:8]
    
    def _check_backward_compat(self, new_schema: dict, old_schema: dict) -> tuple:
        """Check if new schema can read data written with old schema (BACKWARD)."""
        old_fields = {f['name']: f for f in old_schema.get('fields', [])}
        new_fields = {f['name']: f for f in new_schema.get('fields', [])}
        
        # New schema removes a field without default → old data can't be read
        for fname, field in old_fields.items():
            if fname not in new_fields:
                # Removed field - OK only if it had a default in old schema  
                pass  # Simplified: assume removals are OK
        
        # New schema adds a field WITHOUT a default → can't read old data (backward FAIL)
        violations = []
        for fname, field in new_fields.items():
            if fname not in old_fields and 'default' not in field:
                violations.append(f"New field '{fname}' has no default value → BACKWARD INCOMPATIBLE")
        
        return len(violations) == 0, violations
    
    def register(self, subject: str, schema: dict) -> tuple:
        """Register a schema and return (schema_id, version)."""
        if subject not in self.schemas:
            self.schemas[subject] = []
        
        # Check compatibility with latest version
        if self.schemas[subject] and self.compatibility in ('BACKWARD', 'FULL'):
            latest = self.schemas[subject][-1]
            ok, issues = self._check_backward_compat(schema, latest['schema'])
            if not ok:
                return None, None, issues
        
        # Register
        schema_hash = self._schema_hash(schema)
        if schema_hash not in self.schema_id_map:
            schema_id = self.next_id
            self.next_id += 1
            self.schema_id_map[schema_hash] = schema_id
        else:
            schema_id = self.schema_id_map[schema_hash]
        
        version = len(self.schemas[subject]) + 1
        self.schemas[subject].append({'schema': schema, 'id': schema_id, 'version': version})
        return schema_id, version, []
    
    def serialize(self, schema_id: int, data: dict) -> bytes:
        """Simulate Avro serialization: magic_byte + schema_id (4 bytes) + data"""
        payload = json.dumps(data).encode()
        # Confluent wire format: 0x00 + 4-byte schema_id + avro_bytes
        header = struct.pack('>bI', 0, schema_id)
        return header + payload
    
    def deserialize(self, raw_bytes: bytes) -> tuple:
        """Deserialize Confluent wire format message."""
        magic, schema_id = struct.unpack('>bI', raw_bytes[:5])
        data = json.loads(raw_bytes[5:])
        return schema_id, data


# Demo Schema Evolution
registry = SchemaRegistry(compatibility='BACKWARD')

# Version 1: Original user schema
schema_v1 = {
    "type": "record", "name": "User",
    "fields": [
        {"name": "id", "type": "long"},
        {"name": "name", "type": "string"},
    ]
}

# Version 2: Add email WITH default (backward compatible!)
schema_v2 = {
    "type": "record", "name": "User",
    "fields": [
        {"name": "id", "type": "long"},
        {"name": "name", "type": "string"},
        {"name": "email", "type": "string", "default": ""},  # Has default → OK!
    ]
}

# Version 3: Add phone WITHOUT default (backward INCOMPATIBLE!)
schema_v3_bad = {
    "type": "record", "name": "User",
    "fields": [
        {"name": "id", "type": "long"},
        {"name": "name", "type": "string"},
        {"name": "email", "type": "string", "default": ""},
        {"name": "phone", "type": "string"},  # No default → BACKWARD INCOMPATIBLE!
    ]
}

subject = 'user-events-value'

print(f"=== Schema Registry (compatibility={registry.compatibility}) ===")

sid, ver, errs = registry.register(subject, schema_v1)
print(f"\nv1 registration: schema_id={sid}, version={ver}")

sid2, ver2, errs2 = registry.register(subject, schema_v2)
print(f"v2 registration: schema_id={sid2}, version={ver2} (added 'email' with default → OK)")

sid3, ver3, errs3 = registry.register(subject, schema_v3_bad)
if errs3:
    print(f"v3 registration: REJECTED! Reason: {errs3[0]}")
else:
    print(f"v3 registration: schema_id={sid3}, version={ver3}")

# Demonstrate serialization
print("\n=== Wire Format Serialization ===")
user_data = {'id': 42, 'name': 'Alice', 'email': 'alice@example.com'}
raw = registry.serialize(sid2, user_data)
print(f"Serialized bytes: {raw[:20]}... (total {len(raw)} bytes)")
print(f"  Magic byte: 0x{raw[0]:02X}")
print(f"  Schema ID:  {struct.unpack('>I', raw[1:5])[0]}")
print(f"  Payload:    {raw[5:].decode()}")

schema_id_recv, data_recv = registry.deserialize(raw)
print(f"Deserialized: schema_id={schema_id_recv}, data={data_recv}")

---
## 复习要点 (Review Summary)

### 高频考点速记

**1. Topic/Partition/Consumer Group**
- 同一 Group 内：1 个 Partition 只能被 1 个 Consumer 消费
- Consumer 数 > Partition 数 → 多余 Consumer 空闲（浪费）
- 增加并行度 = 增加 Partition 数（不可减少！）
- Partition 内有序，Topic 内无序

**2. 消息语义**
- At-most-once: 先 commit 后 process → 可丢失
- At-least-once: 先 process 后 commit → 可重复（**最常用**）
- Exactly-once: 幂等生产者 + 事务 API → `enable.idempotence=true` + `beginTransaction/commitTransaction`

**3. Replication & ISR**
- ISR = 与 Leader 同步差距在阈值内的副本集合
- `acks=all` + `min.insync.replicas=2` = 最高持久性
- Leader 宕机 → Controller 从 ISR 选新 Leader
- `acks=1` 时 Leader 宕机可能丢 1 条消息（Follower 未同步）

**4. Log Compaction**
- `cleanup.policy=compact` → 只保留每个 key 的最新值
- Tombstone = `value=null` → 标记删除
- 适合 CDC、事件溯源、重建状态

**5. Consumer Lag**
- Lag = LEO - Committed Offset
- Lag 持续增长 → 消费速度 < 生产速度 → 需扩容 Consumer 或优化处理
- 监控：`kafka-consumer-groups.sh --describe`，或 Prometheus + kafka-exporter

**6. Kafka Connect & Schema Registry**
- Connect = 无代码数据集成，Source Connector + Sink Connector
- Schema Registry = Avro Schema 的版本化存储，消息只带 4 字节 schema_id
- BACKWARD compat = 新 Schema 读旧数据（加字段必须有 default）
- FORWARD compat = 旧 Schema 读新数据

---
## 练习 (Exercises)

### 练习 1：Partition 设计 (场景题)

一个电商平台的订单事件 Topic，需要满足：
- 同一用户的订单事件需要有序
- 峰值 QPS 100 万/秒，每条消息 1KB
- 每个 Consumer 实例处理能力 10 万条/秒
- 需要保留 7 天数据

问题：
1. Partition Key 应该选什么？
2. 最少需要多少个 Partition？
3. Consumer Group 最多需要几个 Consumer 实例？
4. 预估存储空间需求？

In [ ]:
# 练习 1 参考计算
qps = 1_000_000           # messages per second
consumer_capacity = 100_000  # messages per consumer per second
msg_size_kb = 1
retention_days = 7

# Q2: Minimum partitions based on consumer throughput
min_partitions = qps / consumer_capacity
print(f"Q1: Partition Key = user_id")
print(f"    同一 user_id → 相同 Partition → 同一用户事件保证有序")
print(f"    高基数（数百万用户 ID）→ 避免热点 Partition")
print()

print(f"Q2: Min Partitions = QPS / Consumer Capacity = {qps:,} / {consumer_capacity:,} = {min_partitions:.0f}")
print(f"    实际建议: 16 或 32（2 的幂次，便于扩容和再分区）")
print()

print(f"Q3: Consumer 实例数 = Partition 数 = {min_partitions:.0f}")
print(f"    （每个 Partition 分配一个 Consumer，多余实例空闲）")
print()

# Q4: Storage
seconds_per_day = 86400
total_msgs = qps * seconds_per_day * retention_days
total_tb = total_msgs * msg_size_kb / (1024**3)
with_replication = total_tb * 3  # replication factor = 3
print(f"Q4: 存储估算")
print(f"    总消息数: {qps:,} * {seconds_per_day:,}s * {retention_days} days = {total_msgs:.2e} 条")
print(f"    原始数据: {total_tb:.1f} TB")
print(f"    含 3x 副本: {with_replication:.1f} TB")
print(f"    (不含压缩，实际 Snappy 压缩可减少 2-3x)")

### 练习 2：消息语义选型 (场景题)

以下场景应选择哪种消息传递语义？说明理由和实现方式：
1. 广告点击计数（可以有少量误差）
2. 银行转账处理（绝对不能重复或丢失）
3. 用户行为日志收集（允许少量丢失，不能影响性能）
4. 库存扣减系统（可以重复处理，但下游需幂等）

In [ ]:
# 练习 2 参考答案
scenarios = [
    {
        'case': '广告点击计数（可少量误差）',
        'semantic': 'At-least-once',
        'config': 'enable.auto.commit=false, 处理后 commitAsync()',
        'reason': '允许少量重复计数（误差可接受），但不能漏计。At-most-once 可能漏计更划不来'
    },
    {
        'case': '银行转账（绝对不能重复/丢失）',
        'semantic': 'Exactly-once',
        'config': 'enable.idempotence=true, beginTransaction/commitTransaction, isolation.level=read_committed',
        'reason': '重复转账或漏转账都会造成资金损失，必须 Exactly-once'
    },
    {
        'case': '用户行为日志（允许丢失，性能优先）',
        'semantic': 'At-most-once',
        'config': 'enable.auto.commit=true, acks=1 (生产者)',
        'reason': '日志允许少量丢失，at-most-once 避免重试，最小化延迟和计算开销'
    },
    {
        'case': '库存扣减（下游幂等）',
        'semantic': 'At-least-once',
        'config': '手动 commitSync()，下游以 order_id 做幂等写入（INSERT IGNORE 或 UPSERT）',
        'reason': '消费者崩溃重启会重消费，下游通过唯一 order_id 去重，整体效果 exactly-once'
    },
]

for s in scenarios:
    print(f"场景: {s['case']}")
    print(f"  语义: {s['semantic']}")
    print(f"  配置: {s['config']}")
    print(f"  原因: {s['reason']}")
    print()

### 练习 3：ISR 故障分析 (概念)

配置：replication.factor=3, min.insync.replicas=2, acks=all

分析以下故障场景：
1. Broker 1（Leader）宕机，Broker 2、3 都在 ISR → 会发生什么？
2. Broker 1（Leader）宕机，只有 Broker 2 在 ISR（Broker 3 落后）→ 结果？
3. Broker 1、2 同时宕机，只剩 Broker 3 → 结果？
4. 为什么 min.insync.replicas 不能设为等于 replication.factor？

In [ ]:
# 练习 3 参考答案
fault_scenarios = [
    {
        'scenario': 'B1宕机，B2/B3都在ISR',
        'result': '正常选主',
        'detail': 'Controller 从 ISR={B2,B3} 中选 B2（或 B3）为新 Leader。无数据丢失，服务短暂中断（秒级）后恢复'
    },
    {
        'scenario': 'B1宕机，只有B2在ISR（B3落后）',
        'result': '正常选主但B3可能缺少最新数据',
        'detail': 'B2 成为新 Leader（ISR 中最新）。B3 重新同步后加入 ISR。B1 恢复后截断落后 B2 LEO 再同步'
    },
    {
        'scenario': 'B1/B2同时宕机，只剩B3',
        'result': '生产者报错 NotEnoughReplicasException',
        'detail': 'ISR 只剩 {B3}，size=1 < min.insync.replicas=2。acks=all 时所有 produce 失败。读操作仍可从 B3 进行'
    },
    {
        'scenario': 'min.insync.replicas == replication.factor 的问题',
        'result': '任意一个 Broker 宕机都导致生产者报错',
        'detail': '若 RF=3, min.insync=3：任何 1 个 Broker 下线 → ISR.size=2 < 3 → 写入全部失败。可用性极差。通常设 min.insync = RF - 1'
    },
]

for i, s in enumerate(fault_scenarios, 1):
    print(f"Q{i}: {s['scenario']}")
    print(f"  结果: {s['result']}")
    print(f"  分析: {s['detail']}")
    print()

### 练习 4：Consumer Lag 排查 (场景题)

你收到告警：consumer group `order-processor` 的 lag 从 0 突然增长到 50,000，且在持续增长。

请描述排查步骤和可能原因。

### 练习 5：Schema Evolution 设计 (设计题)

当前 User Schema（v1）：`{id: long, name: string}`

需求变更：
1. 加入 `email` 字段（所有消费者都需要读取）
2. 将 `name` 字段拆分为 `first_name` 和 `last_name`
3. 删除已废弃的 `email` 字段（之前已全部迁移完）

请设计每步的 Schema 变更方案，说明兼容性考量。

In [ ]:
# 练习 4 参考答案：Consumer Lag 排查步骤
print("=== Consumer Lag 排查步骤 ===")
steps = [
    "1. 确认范围: kafka-consumer-groups.sh --describe → 是全部 Partition 还是特定 Partition lag 大？",
    "2. 检查 Consumer 健康: 查看 Consumer 实例数是否减少（是否有 Rebalance 正在发生）",
    "3. 对比生产速率: 查看 Topic 的 messages-in-rate 是否突增（如营销活动触发流量洪峰）",
    "4. 检查处理速率: Consumer 的 records-consumed-rate 是否下降（下游 DB/API 变慢？）",
    "5. 查看应用日志: 是否有大量 Exception、GC 暂停、超时错误",
    "6. 查看系统指标: CPU/内存/网络，Consumer 机器是否资源耗尽",
    "7. 查看 Rebalance 日志: 频繁 Rebalance 会导致消费周期性停顿",
]
for step in steps:
    print(f"  {step}")

print()
print("=== 可能原因 ===")
causes = {
    '所有 Partition Lag 均增': '生产速率 > 消费速率，需扩容 Consumer 或优化处理逻辑',
    '单个 Partition Lag 大': '该 Partition 的 Consumer 实例挂掉或 Rebalance 后未恢复，或热点 Partition',
    'Lag 突增后稳定': '短暂流量峰值，消费者正在追赶，可能自行恢复',
    'Lag 持续线性增长': '系统性瓶颈，必须扩容（增加 Consumer 实例或 Partition 数）',
    'Lag 时高时低': '下游系统不稳定（GC、网络抖动），检查下游依赖',
}
for symptom, diagnosis in causes.items():
    print(f"  [{symptom}]")
    print(f"    → {diagnosis}")

print()

# 练习 5 参考答案：Schema Evolution
print("=== Schema Evolution 方案 ===")

schema_v1 = {'fields': [{'name': 'id', 'type': 'long'}, {'name': 'name', 'type': 'string'}]}
# Step 1: Add email with default (BACKWARD compatible)
schema_v2 = {'fields': [
    {'name': 'id', 'type': 'long'},
    {'name': 'name', 'type': 'string'},
    {'name': 'email', 'type': 'string', 'default': ''},  # default required!
]}
# Step 2: Add first/last name with defaults, keep name as deprecated
schema_v3 = {'fields': [
    {'name': 'id', 'type': 'long'},
    {'name': 'name', 'type': 'string', 'default': ''},  # deprecated, keep with default
    {'name': 'email', 'type': 'string', 'default': ''},
    {'name': 'first_name', 'type': 'string', 'default': ''},
    {'name': 'last_name', 'type': 'string', 'default': ''},
]}
# Step 3: Remove email (only after ALL consumers migrated, use FORWARD compat window)
schema_v4 = {'fields': [
    {'name': 'id', 'type': 'long'},
    {'name': 'name', 'type': 'string', 'default': ''},
    {'name': 'first_name', 'type': 'string', 'default': ''},
    {'name': 'last_name', 'type': 'string', 'default': ''},
]}

evolutions = [
    ('v1→v2', 'BACKWARD compatible', '加 email 字段并设 default=""，旧消费者可忽略新字段，新消费者可读旧消息（填 default）'),
    ('v2→v3', 'BACKWARD compatible', '加 first/last_name 字段并设 default，旧 name 字段暂保留并加 default（允许新生产者不填）'),
    ('v3→v4', 'FORWARD compatible', '删除 email 字段前确认所有消费者已迁移。删除后旧消费者（仍期望 email）会读到 null/default'),
]

for step, compat, note in evolutions:
    print(f"  {step}: [{compat}]")
    print(f"    {note}")